In [12]:
from openai import OpenAI
import os
import requests

client = OpenAI(
  api_key=os.environ["OPENAI_API_KEY"]
)
city=input("Enter the name of a city: ")
response = client.chat.completions.create(
  model="gpt-5-nano",
  messages=[
    {"role": "user", "content": "Describe the city of {} in a few sentences.".format(city)}

  ]
)
def fetch_weather(city_name: str) -> dict:
    """
    pull the current/forecast data for `city_name` from wttr.in
    (no API key required; returns JSON).
    """
    url = f"https://wttr.in/{city_name}?format=j1"
    r = requests.get(url)
    r.raise_for_status()
    return r.json()

def summarize_weather(city_name: str) -> str:
    """
    two‑step agent: 1) fetch data, 2) ask the LLM to summarise it.
    """
    weather_data = fetch_weather(city_name)
    prompt = (
        "Here is raw weather data in JSON for a city. "
        "Please summarise the current temperature/humidity and the "
        "forecast for the next few days in a couple of sentences.\n\n"
        f"{weather_data}"
    )
    resp = client.chat.completions.create(
        model="gpt-5-nano",
        messages=[
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": prompt},
        ],
    )
    return resp.choices[0].message.content

# run the agent for the existing `city` variable
summary = summarize_weather(city)
print(f"Weather summary for {city}:\n{summary}")
print(response.choices[0].message.content)


Weather summary for kathmandu:
In Kathmandu, the current temperature is 20°C with 56% humidity. The next couple of days look sunny, with highs near 27–28°C and overnight lows around 9–10°C.
Kathmandu is Nepal’s capital and largest city, set in the Kathmandu Valley surrounded by hills and the Himalayas. It’s a centuries-old cultural hub filled with temples, shrines, and historic squares—notably Kathmandu Durbar Square, Swayambhunath (the Monkey Temple), and Boudhanath Stupa. The city blends ancient religious life with a busy, colorful urban scene—narrow streets, vibrant markets, and festivals alongside growing cafés and shops. It also serves as the gateway to the Himalayas, a base for trekkers and climbers heading to nearby trails.
